In [13]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import CharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_openai import OpenAIEmbeddings
from dotenv import load_dotenv, find_dotenv

import os

print("Cartella corrente:", os.getcwd())
print("File presenti:", os.listdir())

text_splitter = CharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=0
)

load_dotenv(find_dotenv())
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

print("Chiave caricata:", OPENAI_API_KEY is not None)

Cartella corrente: c:\Users\sandv\OneDrive\Plocha\IVAN\IRON HACK\.ipynb dal corso\w7\mercoledi\lab-agent-vector-store
File presenti: ['.env', '.git', '.gitignore', '.venv', 'datasets', 'lab-agent-vector-store prof.ipynb', 'lab-agent-vector-store.ipynb', 'README.md']
Chiave caricata: True


In [16]:
sonnet_loader = TextLoader("./datasets/sonnets.txt", encoding="utf-8")
sonnet_documents = sonnet_loader.load()
embeddings = OpenAIEmbeddings(api_key=OPENAI_API_KEY)

sonnet_texts = text_splitter.split_documents(sonnet_documents)

sonnet_db = Chroma.from_documents(
    sonnet_texts,
    embeddings,
    collection_name="shakespeare-sonnets"
)

from langchain.chains import RetrievalQA

from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="gpt-4.1-mini",
    temperature=0,
    api_key=OPENAI_API_KEY
)

sonnet_qa = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=sonnet_db.as_retriever()
)

In [17]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import CharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain.chains import RetrievalQA
from dotenv import load_dotenv, find_dotenv
import os

# Load API key
load_dotenv(find_dotenv())
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

if not OPENAI_API_KEY:
    raise ValueError("OPENAI_API_KEY non trovata nel file .env")

# Text splitter
text_splitter = CharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=0
)

# Load Shakespeare's sonnets
sonnet_loader = TextLoader(
    "./datasets/sonnets.txt",
    encoding="utf-8"
)
sonnet_documents = sonnet_loader.load()

print(f"Documenti caricati: {len(sonnet_documents)}")
print(f"Cartella corrente: {os.getcwd()}")

Documenti caricati: 1
Cartella corrente: c:\Users\sandv\OneDrive\Plocha\IVAN\IRON HACK\.ipynb dal corso\w7\mercoledi\lab-agent-vector-store


In [18]:
# Create embeddings
embeddings = OpenAIEmbeddings(api_key=OPENAI_API_KEY)

# Split documents
sonnet_texts = text_splitter.split_documents(sonnet_documents)

# Create vector store
sonnet_db = Chroma.from_documents(
    sonnet_texts,
    embeddings,
    collection_name="shakespeare-sonnets"
)

# Initialize language model
llm = ChatOpenAI(
    model="gpt-4.1-mini",
    temperature=0,
    api_key=OPENAI_API_KEY
)

# Create retrieval QA chain
sonnet_qa = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=sonnet_db.as_retriever()
)

print(f"Chunk creati: {len(sonnet_texts)}")
print("Vector store e RetrievalQA creati correttamente.")

Chunk creati: 154
Vector store e RetrievalQA creati correttamente.


In [23]:
from langchain_community.document_loaders import WebBaseLoader
from langchain.agents import Tool
from langchain.chains import RetrievalQA

# 1. Caricare le FAQ di Ruff
ruff_loader = WebBaseLoader("https://docs.astral.sh/ruff/faq/")
ruff_documents = ruff_loader.load()

# 2. Creare i chunk e il vector store
ruff_texts = text_splitter.split_documents(ruff_documents)

ruff_db = Chroma.from_documents(
    ruff_texts,
    embeddings,
    collection_name="ruff-faq"
)

# 3. Creare la chain di Ruff
ruff = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=ruff_db.as_retriever()
)

# 4. Definire i tool
tools = [
    Tool(
        name="Shakespeare Sonnets QA",
        func=sonnet_qa.run,
        description=(
            "Answers questions about Shakespeare's sonnets, "
            "their themes, imagery, and poetic content."
        ),
    ),
    Tool(
        name="Ruff QA System",
        func=ruff.run,
        description=(
            "Answers questions about Ruff, the Python linter, "
            "including rules, configuration, and usage."
        ),
    ),
]

print("Chunk Ruff creati:", len(ruff_texts))
print("Tool disponibili:", [tool.name for tool in tools])

Created a chunk of size 2209, which is longer than the specified 1000
Created a chunk of size 3190, which is longer than the specified 1000
Created a chunk of size 1017, which is longer than the specified 1000
Created a chunk of size 2313, which is longer than the specified 1000


Chunk Ruff creati: 23
Tool disponibili: ['Shakespeare Sonnets QA', 'Ruff QA System']


In [24]:
from langchain.agents import initialize_agent, AgentType

agent = initialize_agent(
    tools=tools,
    llm=llm,
    agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    verbose=True,
)

print("Agente creato correttamente.")

Agente creato correttamente.


C:\Users\sandv\AppData\Local\Temp\ipykernel_6196\312200217.py:3: LangChainDeprecationWarning: The function `initialize_agent` was deprecated in LangChain 0.1.0 and will be removed in 1.0. Use Use new agent constructor methods like create_react_agent, create_json_agent, create_structured_chat_agent, etc. instead.
  agent = initialize_agent(


In [25]:
questions = [
    # 1. Solo Shakespeare
    "What recurring themes appear in Shakespeare's sonnets?",

    # 2. Solo Ruff
    "How can I configure Ruff to check my Python code?",

    # 3. Domanda che richiede potenzialmente entrambi i tool
    "I want to write a Python script that counts recurring themes in Shakespeare's sonnets. What themes could I look for, and how can I use Ruff to check the script?",

    # 4. Domanda fuori argomento
    "What is the capital of Japan?",
]

for i, question in enumerate(questions, start=1):
    print(f"\n{'=' * 60}")
    print(f"QUESTION {i}: {question}")
    print("=" * 60)

    try:
        answer = agent.run(question)
        print("\nANSWER:", answer)
    except Exception as e:
        print(f"\nERROR: {type(e).__name__}: {e}")

C:\Users\sandv\AppData\Local\Temp\ipykernel_6196\17277703.py:21: LangChainDeprecationWarning: The method `Chain.run` was deprecated in langchain 0.1.0 and will be removed in 1.0. Use invoke instead.
  answer = agent.run(question)
Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")



QUESTION 1: What recurring themes appear in Shakespeare's sonnets?
Thought: To answer the question about recurring themes in Shakespeare's sonnets, I should use the Shakespeare Sonnets QA tool to get detailed information about the themes present in his sonnets.

Action: Shakespeare Sonnets QA
Action Input: What are the recurring themes in Shakespeare's sonnets?

Observation: The recurring themes in Shakespeare's sonnets include:

1. **Love:** Many sonnets explore various facets of love, including romantic love, unrequited love, and the enduring nature of true love. For example, Sonnet 77 reflects on the poet's consistent focus on love as his central subject.

2. **Time and Change:** Shakespeare often contemplates the passage of time, its effects on beauty and life, and the desire to preserve moments through poetry.

3. **Beauty:** The sonnets frequently address physical beauty, its transient nature, and the attempt to immortalize it through verse.

4. **Mortality and Immortality:** Th

Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")


Final Answer: The recurring themes in Shakespeare's sonnets include love (in its various forms), the passage of time and change, beauty and its transience, mortality and the hope for immortality through poetry, the power of art and poetry itself, and identity and self-reflection. These themes are interwoven to explore the complexities of human experience.

> Finished chain.

ANSWER: The recurring themes in Shakespeare's sonnets include love (in its various forms), the passage of time and change, beauty and its transience, mortality and the hope for immortality through poetry, the power of art and poetry itself, and identity and self-reflection. These themes are interwoven to explore the complexities of human experience.

QUESTION 2: How can I configure Ruff to check my Python code?
Thought: The question is about configuring Ruff, the Python linter, to check Python code. I should provide information on how to set up Ruff, including configuration files and basic usage.
Action: Ruff QA Sy

Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")


Final Answer:  
To configure Ruff to check your Python code, follow these steps:

1. **Install Ruff**  
   Use pip or another package manager:
   ```bash
   pip install ruff
   ```

2. **Create a configuration file**  
   Ruff supports configuration via `pyproject.toml` or `ruff.toml` in your project directory. For example, create a `pyproject.toml` with:
   ```toml
   [tool.ruff]
   line-length = 88
   select = ["E", "F", "W"]  # Select specific error codes to check
   ignore = ["E501"]         # Ignore specific error codes
   ```
   This sets the max line length and which linting rules to check or ignore.

3. **Run Ruff**  
   Run Ruff on your code directory or files:
   ```bash
   ruff path/to/your/code
   ```

4. **Additional options**  
   - Ruff supports import sorting options (some isort features).  
   - You can enforce docstring conventions by adding a `convention` setting.  
   - Ruff can lint Jupyter Notebooks via `nbqa`.

5. **Linting Jupyter Notebooks**  
   After installi

Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")



Observation: The recurring themes in Shakespeare's sonnets include:

1. **Love**: Many sonnets explore various facets of love, including romantic love, unrequited love, and the enduring nature of love. For example, Sonnet 77 reflects on love as a constant subject, with the poet always writing about "you and love."

2. **Time and Change**: Shakespeare often contemplates the passage of time, its effects on beauty and life, and the desire to preserve moments through poetry.

3. **Beauty**: The celebration and preservation of physical and inner beauty is a common theme, often linked with the effects of time.

4. **Mortality and Immortality**: The sonnets frequently address human mortality and the poet's attempt to achieve immortality through verse.

5. **Art and Poetry**: Shakespeare reflects on the power of poetry itself to immortalize subjects and emotions, as seen in the idea of "dressing old words new" in Sonnet 77.

These themes intertwine to explore human experience, emotion, and th

In [26]:
from langchain.agents import Tool, initialize_agent, AgentType

tools_direct = [
    Tool(
        name="Shakespeare Sonnets QA",
        func=sonnet_qa.run,
        description=(
            "Answers questions about Shakespeare's sonnets, "
            "their themes, imagery, and poetic content."
        ),
        return_direct=True,
    ),
    Tool(
        name="Ruff QA System",
        func=ruff.run,
        description=(
            "Answers questions about Ruff, the Python linter, "
            "including rules, configuration, and usage."
        ),
        return_direct=True,
    ),
]

agent_direct = initialize_agent(
    tools=tools_direct,
    llm=llm,
    agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    verbose=True,
)

print("Agente con return_direct=True creato correttamente.")

Agente con return_direct=True creato correttamente.


In [27]:
question = "What recurring themes appear in Shakespeare's sonnets?"

print("\n" + "=" * 60)
print("AGENTE PREDEFINITO: return_direct=False")
print("=" * 60)

answer_default = agent.run(question)
print("\nRISPOSTA FINALE:\n", answer_default)


print("\n" + "=" * 60)
print("AGENTE DIRETTO: return_direct=True")
print("=" * 60)

answer_direct = agent_direct.run(question)
print("\nRISPOSTA FINALE:\n", answer_direct)

Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")



AGENTE PREDEFINITO: return_direct=False
Thought: To answer the question about recurring themes in Shakespeare's sonnets, I should use the Shakespeare Sonnets QA tool to get detailed information about the themes present in his sonnets.

Action: Shakespeare Sonnets QA
Action Input: What are the recurring themes in Shakespeare's sonnets?

Observation: The recurring themes in Shakespeare's sonnets include:

1. **Love:** Many sonnets explore various facets of love, including romantic love, unrequited love, and the enduring nature of true love. For example, Sonnet 77 reflects on the poet's consistent focus on love as his central subject.

2. **Time and Change:** Shakespeare often contemplates the passage of time, its effects on beauty and life, and the desire to preserve moments through poetry.

3. **Beauty:** The sonnets frequently address physical beauty, its transient nature, and the attempt to immortalize it through verse.

4. **Mortality and Immortality:** There is a tension between hu

Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")


Final Answer: The recurring themes in Shakespeare's sonnets include love (in its various forms), the passage of time and change, beauty and its transience, mortality versus immortality, the power of art and poetry to preserve emotions and beauty, and identity and self-reflection. These themes are intricately woven throughout the sonnets, offering a profound exploration of human experience.

> Finished chain.

RISPOSTA FINALE:
 The recurring themes in Shakespeare's sonnets include love (in its various forms), the passage of time and change, beauty and its transience, mortality versus immortality, the power of art and poetry to preserve emotions and beauty, and identity and self-reflection. These themes are intricately woven throughout the sonnets, offering a profound exploration of human experience.

AGENTE DIRETTO: return_direct=True
Thought: To answer the question about recurring themes in Shakespeare's sonnets, I should use the Shakespeare Sonnets QA tool to get detailed information 

In [28]:
question = (
    "I want to write a Python script that counts recurring themes "
    "in Shakespeare's sonnets. What themes could I look for, "
    "and how can I use Ruff to check the script?"
)

answer_direct_multi = agent_direct.run(question)
print("\nRISPOSTA:\n", answer_direct_multi)

Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")


Thought: To write a Python script that counts recurring themes in Shakespeare's sonnets, I first need to identify what the common themes are in the sonnets. Then, I can explain how to use Ruff to check the Python script for linting and style issues.

Action: Shakespeare Sonnets QA
Action Input: What are the recurring themes in Shakespeare's sonnets?

Observation: The recurring themes in Shakespeare's sonnets include:

1. **Love:** Many sonnets explore various aspects of love, including romantic love, unrequited love, and the enduring nature of true love. For example, Sonnet 77 reflects on love as a constant subject, with the poet continually writing about the same beloved.

2. **Time and Mortality:** Shakespeare often contemplates the passage of time, aging, and the inevitability of death, contrasting them with the desire for immortality through poetry.

3. **Beauty:** The sonnets frequently address physical beauty, its fleeting nature, and the attempt to preserve it through verse.

4.

In [ ]:
## Reflection

The agent uses the tools' names and descriptions to decide which tool to call. The Shakespeare and Ruff tools worked for questions related to their respective topics. For an unrelated question, the agent answered directly using the language model's general knowledge without calling either tool.

The multi-hop question revealed a limitation: with the default configuration (`return_direct=False`), the agent called both tools but repeatedly returned to the Shakespeare tool and eventually reached the iteration limit without completing the answer.

With `return_direct=True`, the tool's output is returned directly, without further processing by the agent. This is useful for straightforward questions but can prevent the agent from combining information from multiple tools. In our test, the agent returned the Shakespeare result without consulting Ruff.

Therefore, the default behavior is more suitable when a question requires multiple tools, although the agent's reasoning and tool selection may still need improvement. `return_direct=True` is more appropriate when the result of a single tool is sufficient.